In [0]:
from pyspark.sql.functions import to_date, trim, initcap,col,split, col,round,datediff,when,current_date,monotonically_increasing_id

employee_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_employee")

text_columns = ["first_name", "last_name"]
for col_name in text_columns:
    employee_df = employee_df.withColumn(col_name, (trim(col_name)))

employee_df = employee_df.withColumn("base_salary", round(col("base_salary"), 2))

employee_df = employee_df.withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy HH:mm")) \
       .withColumn("termination_date", to_date(col("termination_date"), "dd-MM-yyyy HH:mm"))\
       .withColumn("tenure_days",
        datediff(
            when(col("termination_date").isNotNull(), col("termination_date"))
            .otherwise(current_date()),
            col("hire_date"))) \
       .withColumn("employee_sk",monotonically_increasing_id()).withColumnRenamed("is_active", "employee_is_active")


employee_df = employee_df.withColumn("termination_date",
    when(trim(col("termination_date")) == "", None)
    .otherwise(col("termination_date"))
)

employee_df.write.format("delta").mode("overwrite").saveAsTable("global_tech_sliver.transformed_data_silver.tf_employee")

display(employee_df)
